In [ ]:
# ================================
# Environment Setup
# ================================
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [ ]:
# ================================
# Image Parameters
# ================================
IMG_HEIGHT = 160 #96
IMG_WIDTH = 160 #96
CHANNELS = 3
NUM_CLASSES = 7
BATCH_SIZE = 24

In [ ]:
# ================================
# Dataset Paths (Colab)
# ================================
TRAIN_DIR = "/content/drive/MyDrive/Colab Notebooks/Indian Currency dataset/train"
VAL_DIR   = "/content/drive/MyDrive/Colab Notebooks/Indian Currency dataset/valid"
TEST_DIR  = "/content/drive/MyDrive/Colab Notebooks/Indian Currency dataset/test"

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=[0.6, 1.4],
    shear_range=0.15,
    fill_mode="nearest"
    )

val_gen   = ImageDataGenerator(rescale=1./255)
test_gen  = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="sparse"
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)


Found 3173 images belonging to 7 classes.
Found 315 images belonging to 7 classes.
Found 83 images belonging to 7 classes.


In [ ]:
# ================================
# MobileNetV2 Model
# ================================
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = True  # before i Freeze pretrained layers, now enable fine tuining
for layer in base_model.layers[:-80]: #-40
    layer.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=1e-5), # before i was using 0.0001
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,423 (8.93 MB)

 Trainable params: 2,150,151 (8.20 MB)

 Non-trainable params: 190,272 (743.25 KB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]


In [ ]:
# ================================
# Training
# ================================
history = model.fit(
    train_data,
    epochs=15,
    validation_data=val_data,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 2431s 18s/step - accuracy: 0.1720 - loss: 2.3623 - val_accuracy: 0.1524 - val_loss: 2.1218 - learning_rate: 1.0000e-05
Epoch 2/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.2565 - loss: 1.9059 - val_accuracy: 0.2476 - val_loss: 1.8757 - learning_rate: 1.0000e-05
Epoch 3/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - accuracy: 0.3584 - loss: 1.7047 - val_accuracy: 0.3524 - val_loss: 1.7031 - learning_rate: 1.0000e-05
Epoch 4/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 153s 1s/step - accuracy: 0.4293 - loss: 1.5728 - val_accuracy: 0.4127 - val_loss: 1.5599 - learning_rate: 1.0000e-05
Epoch 5/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 135s 1s/step - accuracy: 0.4976 - loss: 1.4093 - val_accuracy: 0.4571 - val_loss: 1.4231 - learning_rate: 1.0000e-05
Epoch 6/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.5532 - loss: 1.2679 - val_accuracy: 0.5619 - val_loss: 1.2754 - learning_rate: 1.0000e-05
Epoch 7/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 135s 1s/step - a

In [ ]:
model.export("/content/drive/MyDrive/mobilenet_currency_detection")


Saved artifact at '/content/drive/MyDrive/mobilenet_currency_detection'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  138075767265168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232952016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232952400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073234472400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232950672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073234472592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232951056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232953168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232952784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138073232952592: TensorSpec(shape=(), dtype=t

In [ ]:
import json

class_names = [None] * len(train_data.class_indices)
for name, index in train_data.class_indices.items():
    class_names[index] = name

with open("/content/drive/MyDrive/class_names.json", "w") as f:
    json.dump(class_names, f)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Reset generator (VERY IMPORTANT)
test_data.reset()

# Predict probabilities
y_pred_probs = model.predict(test_data, verbose=1)

# Convert probabilities to class indices
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels
y_true = test_data.classes

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")

print("\n📊 Model Performance Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

class_names = list(test_data.class_indices.keys())

print("\n📋 Detailed Classification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))


4/4 ━━━━━━━━━━━━━━━━━━━━ 46s 13s/step

📊 Model Performance Metrics
Accuracy  : 0.8675
Precision : 0.8752
Recall    : 0.8675
F1-score  : 0.8674

📋 Detailed Classification Report:

              precision    recall  f1-score   support

          10     1.0000    0.7143    0.8333         7
         100     0.7500    0.8571    0.8000        14
          20     0.8333    0.8333    0.8333        12
         200     0.9333    0.9333    0.9333        15
        2000     0.8462    1.0000    0.9167        11
          50     0.8333    0.7692    0.8000        13
         500     1.0000    0.9091    0.9524        11

    accuracy                         0.8675        83
   macro avg     0.8852    0.8595    0.8670        83
weighted avg     0.8752    0.8675    0.8674        83

